# Лекция 3 · Машинное обучение в вычислительной физике

Выполняйте ячейки сверху вниз. Номера и порядок разделов соответствуют слайдам.

Необходимые библиотеки: `numpy matplotlib torch`.

Перед первым запуском выполните в терминале среды курса:
```bash
python -m pip install numpy matplotlib torch
```
Обучение суррогатной модели и PINN может занять некоторое время.


## Содержание
- [3.1 Персептрон и классификация](#section-3-1)
- [3.2 Нейронная сеть и прямой проход](#section-3-2)
- [3.3 Обучение и проверка модели](#section-3-3)
- [3.4 Прогноз теплопроводности по данным](#section-3-4)
- [3.5 Температурное поле из уравнения теплопроводности](#section-3-5)


<a id="section-3-1"></a>

## 3.1 Персептрон и классификация


In [ ]:
# Строки X — четыре двоичных входа; y задаёт таблицу истинности AND.
# X 的各行是四种二值输入，y 给出 AND 的真值表。
import numpy as np
import matplotlib.pyplot as plt
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0,0,0,1])
print("Линейные отклики:", X @ [2,1]-1.5)
print("AND:", (X @ [1,1]-1.5 > 0).astype(int))

### Один шаг обучения персептрона


In [ ]:
# Правило персептрона корректирует веса и смещение по ошибке target − prediction.
# 感知机按 target − prediction 的误差更新权重和偏置。
w = np.array([1.,1.]); b = -0.5
x, target, eta = np.array([1.,0.]), 0, 0.5
prediction = int(w @ x+b > 0)
error = target-prediction
w += eta*error*x
b += eta*error
print(w, b, "Новый прогноз:", int(w @ x+b > 0))
assert np.allclose(w,[0.5,1]) and b == -1
assert np.array_equal((X @ w+b > 0).astype(int), y)

### И и исключающее ИЛИ
У XOR точки одного класса лежат на диагонали; одна прямая их не разделяет. В следующем разделе добавим скрытый слой.


In [ ]:
# Для AND существует разделяющая прямая; для XOR одной прямой недостаточно.
# AND 可用一条直线分开；XOR 无法用单一直线分开。
fig, axes = plt.subplots(1,2,figsize=(8,3))
for ax, labels, title in zip(axes,[y,np.array([0,1,1,0])],["AND","XOR"]):
    ax.scatter(X[:,0],X[:,1],c=labels,cmap="coolwarm",s=90)
    ax.set(xlabel="x1",ylabel="x2",title=title,xlim=(-0.2,1.2),ylim=(-0.2,1.2))
axes[0].plot([0.5,1.2],[1.,0.3],"--")
plt.tight_layout()
plt.show()

<a id="section-3-2"></a>

## 3.2 Нейронная сеть и прямой проход


In [ ]:
# Загружаем библиотеки для вычисления слоёв и построения графиков.
# 导入用于逐层计算和绘图的库。
import numpy as np
import matplotlib.pyplot as plt

### Послойный расчёт в NumPy


In [ ]:
# Два скрытых порога выделяют сумму входов между 0 и 2, реализуя XOR.
# 两个隐藏阈值筛选出输入和大于 0 且小于 2 的情形，从而实现 XOR。
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
W = np.array([[1, -1], [1, -1]])
b = np.array([-0.5, 1.5])
H = (X @ W + b > 0).astype(int)
y_hat = (H @ [1, 1] - 1.5 > 0).astype(int)
print(y_hat)

### Как получить прогноз для нового входа


In [ ]:
# Прямой проход: линейное преобразование, ReLU, затем выходной слой.
# 前向传播依次执行线性变换、ReLU 和输出层计算。
import numpy as np
x = np.array([1.0, 2.0, 4.0])
z = 0.5*x
h = np.maximum(0, z)
y_hat = 0.5*h
print(y_hat)                       # [0.25 0.5  1.  ]

### Активации и потеря одного образца


In [ ]:
# Сравниваем активации и вычисляем половину квадрата ошибки одного примера.
# 比较激活函数，并计算单个样本误差平方的一半。
z = np.linspace(-4,4,200)
plt.plot(z,(z>0).astype(float),label="Ступенчатая")
plt.plot(z,np.maximum(0,z),label="ReLU")
plt.plot(z,1/(1+np.exp(-z)),label="Сигмоида")
plt.legend()
plt.show()
x,y = 2.,1.
w,b,v,c = 0.5,0.,0.5,0.
h = max(0,w*x+b)
prediction = v*h+c
loss = (prediction-y)**2/2
print(h,prediction,loss)
assert loss == 0.125

<a id="section-3-3"></a>

## 3.3 Обучение и проверка модели


In [ ]:
# Задаём общие цвета для последующих графиков обучения и проверки.
# 设置后续训练和检验图中使用的统一颜色。
import numpy as np
import matplotlib.pyplot as plt
blue, red = "#356AE6", "#D94F65"

### Обратное распространение и обновление


In [ ]:
# Для положительного входа ReLU производная равна 1; градиенты считаем по цепному правилу.
# ReLU 输入为正时导数为 1；按链式法则计算梯度。
x,y = 2.,1.
p = np.array([0.5,0.,0.5,0.])  # w,b,v,c
w,b,v,c = p
h = max(0,w*x+b)
prediction = v*h+c
error = prediction-y
gradients = np.array([error*v*x, error*v, error*h, error])
print("Градиенты:",gradients)
p -= 0.1*gradients
w,b,v,c = p
prediction = v*max(0,w*x+b)+c
print("Новые параметры:",p,"Прогноз:",prediction,"Потеря:",(prediction-y)**2/2)
assert np.allclose(p,[0.55,0.025,0.55,0.05])
assert np.isclose(prediction,0.66875)

### Проверка четырёх производных в PyTorch


In [ ]:
# requires_grad включает учёт операций; backward вычисляет градиенты параметров.
# requires_grad 开启运算记录；backward 计算参数梯度。
import torch
p = torch.tensor([0.5, 0.0, 0.5, 0.0], requires_grad=True)
w, b, v, c = p
h = torch.relu(w*2 + b)
prediction = v*h + c
loss = (prediction - 1)**2 / 2
loss.backward()
print(p.grad.tolist())

### Наблюдения, обучение и проверка


In [ ]:
# Делим 160 наблюдений на 96 обучающих, 32 валидационных и 32 тестовых.
# 将 160 个样本划分为 96 个训练样本、32 个验证样本和 32 个测试样本。
rng = np.random.default_rng(42)
x = rng.uniform(-2, 2, 160)
y = 2 * x - 1 + rng.normal(0, 0.3, x.size)
indices = rng.permutation(x.size)
train, valid, test = np.split(indices, [96, 128])
x_train, y_train = x[train], y[train]
x_valid, y_valid = x[valid], y[valid]

def half_mse(prediction, target):
    return 0.5 * np.mean((prediction - target)**2)

### Подбор шага обучения по валидации


In [ ]:
# Обучаем линейную модель мини-пакетами и выбираем шаг по валидационной потере.
# 用小批量训练线性模型，并依据验证损失选择学习率。
def fit_line(x_train, y_train, x_valid, y_valid, lr,
             epochs=80, batch_size=16, seed=7):
    rng = np.random.default_rng(seed)
    w, b = 0.0, 0.0
    history = []
    for epoch in range(epochs):
        # Перед каждой эпохой перемешиваем обучающие примеры.
        # 每轮训练前打乱训练样本顺序。
        order = rng.permutation(x_train.size)
        for start in range(0, x_train.size, batch_size):
            ids = order[start:start + batch_size]
            xb, yb = x_train[ids], y_train[ids]
            error = w * xb + b - yb
            # Производные половины MSE по наклону w и смещению b.
            # 计算半均方误差对斜率 w 和偏置 b 的导数。
            dw = np.mean(error * xb)
            db = np.mean(error)
            w -= lr * dw
            b -= lr * db
        train_loss = half_mse(w * x_train + b, y_train)
        valid_loss = half_mse(w * x_valid + b, y_valid)
        history.append([train_loss, valid_loss])
    return w, b, np.array(history)

learning_rates = [0.001, 0.01, 0.1]
runs = [fit_line(x_train, y_train, x_valid, y_valid, lr)
        for lr in learning_rates]
# Выбираем по валидации; тестовые данные здесь не используются.
# 仅依据验证集选择模型，此处不使用测试数据。
best = int(np.argmin([run[2][-1, 1] for run in runs]))
w_fit, b_fit, history = runs[best]
for lr, (_, _, curve) in zip(learning_rates, runs):
    print(f"lr={lr:.3f}, потеря на валидации={curve[-1, 1]:.6f}")
print("Выбранный шаг обучения:", learning_rates[best])
print(f"w={w_fit:.5f}, b={b_fit:.5f}")

### Итоговая проверка на тестовой выборке


In [ ]:
# Тестовая выборка используется после выбора шага обучения для итоговой оценки RMSE.
# 选定学习率后，用测试集进行最终 RMSE 评估。
test_prediction = w_fit * x[test] + b_fit
test_rmse = np.sqrt(np.mean((test_prediction - y[test])**2))
print(f"RMSE на тестовой выборке: {test_rmse:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")
for ids, label, color in [(train, "Обучение", blue), (valid, "Валидация", "#E49A27"),
                           (test, "Тест", red)]:
    axes[0].scatter(x[ids], y[ids], s=18, alpha=0.7, label=label, color=color)
grid = np.linspace(-2, 2, 100)
axes[0].plot(grid, w_fit * grid + b_fit, color="black", label="Модель")
axes[0].set(xlabel="x", ylabel="y"); axes[0].legend(fontsize=9)
axes[1].semilogy(np.arange(1, 81), history[:, 0], label="Обучение", color=blue)
axes[1].semilogy(np.arange(1, 81), history[:, 1], label="Валидация", color=red)
axes[1].set(xlabel="Эпоха", ylabel="Половина MSE"); axes[1].legend()
plt.show()

<a id="section-3-4"></a>

## 3.4 Прогноз теплопроводности по данным


Учебные данные заданы формулой, это не эксперимент. Разбиение 700/150/150; два слоя ReLU по 32 нейрона, пакет 64, 2000 эпох. Выбор весов — по валидации.


In [ ]:
# Фиксируем зерно NumPy и PyTorch для повторяемого учебного эксперимента.
# 固定 NumPy 和 PyTorch 的随机种子，以重现教学实验。
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
blue, red = "#356AE6", "#D94F65"
import copy
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(0)
rng = np.random.default_rng(0)

### Данные
$\kappa(T,L)=\frac{200}{1+0.0005(T-300)^2}\exp(-L/200)$.
$T$: 100–600 K, $L$: 10–500 nm. Масштабируем входы по этим границам, выход делим на 100.


In [ ]:
# Генерируем синтетические данные по формуле, а не из эксперимента.
# 按公式生成合成数据，这些数据并非实验测量值。
def kappa_reference(T, L):
    return 200 / (1 + 0.0005 * (T - 300)**2) * np.exp(-L / 200)

T = rng.uniform(100, 600, 1000)
L = rng.uniform(10, 500, 1000)
features = np.column_stack([T, L])
kappa = kappa_reference(T, L)
# Масштабируем входы по заданным физическим границам, выход делим на 100.
# 按给定物理范围缩放输入，并将输出除以 100。
X = torch.tensor((features - [100, 10]) / [500, 490], dtype=torch.float32)
y = torch.tensor(kappa[:, None] / 100, dtype=torch.float32)
order = rng.permutation(len(X))
# Случайное разбиение: 700 / 150 / 150; множества не пересекаются.
# 随机划分为 700 / 150 / 150 个样本，三个集合互不重叠。
train, valid, test = order[:700], order[700:850], order[850:]
print("Обучение / валидация / тест:", len(train), len(valid), len(test))

### Обучение

Сеть 2 → 32 → 32 → 1, ReLU, Adam. Выбираем эпоху по валидационной выборке; тестовую используем только для итоговой оценки.


In [ ]:
# Сеть 2 → 32 → 32 → 1 обучается на мини-пакетах по 64 примера.
# 采用 2 → 32 → 32 → 1 网络，每个小批量包含 64 个样本。
class KappaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x)

model = KappaNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history, best_loss = [], float("inf")
for epoch in range(2000):
    model.train()
    for batch in torch.tensor(train)[torch.randperm(len(train))].split(64):
        loss = ((model(X[batch]) - y[batch])**2).mean()
        # Очищаем старые градиенты, вычисляем новые и делаем шаг Adam.
        # 清除旧梯度，计算新梯度，再执行 Adam 更新。
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Для проверки отключаем режим обучения и запись градиентов.
    # 验证时切换到评估模式，并关闭梯度记录。
    model.eval()
    with torch.no_grad():
        train_loss = ((model(X[train]) - y[train])**2).mean().item()
        valid_loss = ((model(X[valid]) - y[valid])**2).mean().item()
    history.append([train_loss, valid_loss])
    if valid_loss < best_loss:
        best_loss, best_epoch = valid_loss, epoch + 1
        # Сохраняем независимую копию лучших весов по валидационной ошибке.
        # 按验证误差保存最佳权重的独立副本。
        best_state = copy.deepcopy(model.state_dict())
# Восстанавливаем лучшую эпоху, а не обязательно последнюю.
# 恢复验证表现最佳的轮次，不一定是最后一轮。
model.load_state_dict(best_state)
history = np.asarray(history)

In [ ]:
# При прогнозе применяем то же масштабирование входов и восстанавливаем единицы κ.
# 预测时使用相同的输入缩放，并将 κ 恢复到原始单位。
def predict_kappa(T, L):
    inputs = np.column_stack([np.asarray(T).ravel(), np.asarray(L).ravel()])
    inputs = torch.tensor((inputs - [100, 10]) / [500, 490], dtype=torch.float32)
    with torch.no_grad():
        return model(inputs).numpy().ravel() * 100

prediction = predict_kappa(T[test], L[test])
rmse = np.sqrt(np.mean((prediction - kappa[test])**2))
mae = np.mean(np.abs(prediction - kappa[test]))
print(f"Лучшая эпоха: {best_epoch}; RMSE на тесте: {rmse:.3f}; MAE: {mae:.3f} Вт/(м·К)")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
axes[0].semilogy(np.arange(1, 2001), history[:, 0], label="Обучение", color=blue)
axes[0].semilogy(np.arange(1, 2001), history[:, 1], label="Валидация", color=red)
axes[0].set(xlabel="Эпоха", ylabel="MSE для κ / 100")
axes[0].legend()
axes[1].scatter(kappa[test], prediction, s=14, color=blue)
limits = [min(0, prediction.min()), max(kappa[test].max(), prediction.max())]
axes[1].plot(limits, limits, "--", color=red)
axes[1].set(xlabel="Эталонная κ, Вт/(м·К)", ylabel="Прогноз, Вт/(м·К)")
plt.show()

### За пределами обучающих данных

Фиксируем L = 100 нм и расширяем диапазон температуры. Даже малая ошибка на тестовой выборке не гарантирует правильной экстраполяции. Показываем исходные прогнозы сети, не обрезая отрицательные значения.


In [ ]:
# За пределами диапазона обучения точность не гарантируется; прогнозы не обрезаем.
# 训练范围之外的精度没有保证；保留原始预测值，不做裁剪。
temperature = np.linspace(50, 900, 500)
length = np.full_like(temperature, 100)
fig, ax = plt.subplots(figsize=(9, 3.3), layout="constrained")
ax.axvspan(100, 600, color=blue, alpha=0.08, label="Диапазон обучения")
ax.plot(temperature, kappa_reference(temperature, length), color=blue, label="Формула")
ax.plot(temperature, predict_kappa(temperature, length), color=red, label="Нейросеть")
ax.set(xlabel="T, K", ylabel="κ, Вт/(м·К)")
ax.legend(ncol=3)
plt.show()

<a id="section-3-5"></a>

## 3.5 Температурное поле из уравнения теплопроводности


### Начальный гауссов профиль и точное решение
Уравнение u_t=u_xx на [−5,5]×[0,1], начальное условие exp(−x²/2). На концах задано значение решения бесконечного стержня. Внутреннее точное решение используется только после обучения.


In [ ]:
# Точное решение описывает диффузию гауссова пика: высота падает, ширина растёт.
# 解析解描述高斯峰的扩散：峰值降低，峰宽增大。
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(2026)
dtype = torch.float64

def exact(x,t):
    return torch.exp(-x.square()/(2*(1+2*t)))/torch.sqrt(1+2*t)

x_plot = torch.linspace(-5,5,201,dtype=dtype).reshape(-1,1)
for t_value in [0.,0.25,0.5,1.]:
    plt.plot(x_plot.numpy(),exact(x_plot,torch.tensor(t_value)).numpy(),label=f"t={t_value}")
plt.xlabel("x"); plt.ylabel("u"); plt.legend(); plt.show()
print("t=1: высота пика =", 1/np.sqrt(3), "ширина =", np.sqrt(3))

### Три группы точек
512 внутренних точек, 65 начальных, по 64 на каждом конце. Два скрытых слоя tanh по 24 нейрона.


In [ ]:
# Задаём внутренние, начальные и граничные точки для обучения PINN.
# 设置 PINN 训练所需的内部点、初始点和边界点。
xt = torch.rand((512,2),dtype=dtype)
xt[:,0] = 10*xt[:,0]-5
x0 = torch.linspace(-5,5,65,dtype=dtype).reshape(-1,1)
initial_xt = torch.cat([x0,torch.zeros_like(x0)],dim=1)
initial_u = torch.exp(-x0.square()/2)
tb = torch.linspace(0,1,64,dtype=dtype).reshape(-1,1)
boundary_xt = torch.cat([
    torch.cat([-5*torch.ones_like(tb),tb],dim=1),
    torch.cat([5*torch.ones_like(tb),tb],dim=1)])
# На границе используем значения решения бесконечного стержня.
# 边界条件采用无限长杆解析解在两端的值。
boundary_u = exact(boundary_xt[:,:1],boundary_xt[:,1:])
model = nn.Sequential(nn.Linear(2,24),nn.Tanh(),nn.Linear(24,24),nn.Tanh(),nn.Linear(24,1)).to(dtype)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

def residual(model,xt):
    # Для остатка нужны производные по входам x и t, а не только по весам.
    # 计算方程残差需要对输入 x 和 t 求导，不只是对权重求导。
    xt = xt.detach().requires_grad_(True)
    u = model(xt)
    def grad(a):
        # Сохраняем граф производных для второй производной и обучения весов.
        # 保留导数计算图，以计算二阶导数并训练权重。
        return torch.autograd.grad(a.sum(),xt,create_graph=True)[0]
    du = grad(u)
    ux,ut = du[:,:1],du[:,1:]
    # Остаток уравнения теплопроводности: u_t − u_xx.
    # 热传导方程残差为 u_t − u_xx。
    return ut-grad(ux)[:,:1]

# Проверка остатка точного решения; не обучающая цель внутри области.
# 检查解析解的方程残差，不将内部解析温度用作训练目标。
class ExactSolution(nn.Module):
    def forward(self,xt): return exact(xt[:,:1],xt[:,1:])
assert residual(ExactSolution(),xt).abs().max().item() < 1e-12

### Производные по входам и обновление параметров
Суммируем три среднеквадратичные ошибки без внутренних точных температур.


In [ ]:
# Минимизируем сумму MSE уравнения, начального и граничного условий.
# 最小化方程残差、初始条件和边界条件三项均方误差之和。
history = []
EPOCHS = 3000
for epoch in range(EPOCHS):
    # Внутри области минимизируем остаток уравнения без точных температур.
    # 区域内部仅最小化方程残差，不使用解析温度作为标签。
    equation_loss = residual(model,xt).square().mean()
    initial_loss = (model(initial_xt)-initial_u).square().mean()
    boundary_loss = (model(boundary_xt)-boundary_u).square().mean()
    loss = equation_loss+initial_loss+boundary_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([equation_loss.item(),initial_loss.item(),boundary_loss.item()])
print("Итоговые MSE: уравнение / начальное условие / граница:",history[-1])
plt.semilogy(history)
plt.legend(["Уравнение","Начальное условие","Граничное условие"])
plt.xlabel("Эпоха"); plt.ylabel("MSE"); plt.show()

### Проверка гауссова пика после обучения
Сравните высоту, ширину и ошибку профиля. Результат зависит от числа эпох и случайного зерна.


In [ ]:
# После обучения сравниваем PINN с точным решением и считаем RMSE профиля.
# 训练结束后，将 PINN 与解析解比较，并计算温度分布的 RMSE。
fig,axes = plt.subplots(1,2,figsize=(10,3.5))
for time in [0.,0.5,1.]:
    times = torch.full_like(x_plot,time)
    with torch.no_grad():
        prediction = model(torch.cat([x_plot,times],dim=1))
        reference = exact(x_plot,times)
    axes[0].plot(x_plot.numpy(),reference.numpy(),"--")
    axes[0].plot(x_plot.numpy(),prediction.numpy(),label=f"PINN t={time}")
    axes[1].plot(x_plot.numpy(),(prediction-reference).numpy(),label=f"t={time}")
    print("t:",time,"RMSE:",torch.sqrt((prediction-reference).square().mean()).item())
axes[0].set(xlabel="x",ylabel="u"); axes[1].set(xlabel="x",ylabel="Ошибка")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()